# 06 – Identify EC Cases Citing Other EC Antitrust Cases

This notebook checks which EC antitrust documents (from `ec_antitrust_master.csv`) cite other EC antitrust cases.

**Approach:**
1. Load `ec_antitrust_master.csv`
2. For each EC document with a `document_url`, fetch the full text (PDF or HTML depending on `document_type`)
3. Search the document text for EC case references using the same regex patterns as in `05_identify_cjeu_cases_citing_ec.ipynb`
4. Exclude self-references
5. Export matches to `data/processed/ec_ec_case_matches.csv`

## 1. Imports and Configuration

In [1]:
import io
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
import pypdf

# ── Paths ──────────────────────────────────────────────────────────────────────────────
DATA_DIR       = Path("data/processed")
EC_MASTER_PATH = DATA_DIR / "ec_antitrust_master.csv"
OUTPUT_PATH    = DATA_DIR / "ec_ec_case_matches.csv"

# ── HTTP settings ──────────────────────────────────────────────────────────────────
REQUEST_TIMEOUT = 30   # seconds per request
REQUEST_DELAY   = 1.0  # seconds between requests
MAX_RETRIES     = 2

# ── Context window around each match (characters) ─────────────────────────────────
CONTEXT_CHARS = 200

# document_type values that should be treated as PDF
PDF_DOCUMENT_TYPES = {"case", "decision"}

print("Configuration loaded.")

Configuration loaded.


## 2. Text Normalization and Extraction Helpers

In [2]:
def normalize_text(raw: str) -> str:
    """Normalize Unicode, collapse whitespace, unify hyphens/separators."""
    text = unicodedata.normalize("NFKC", raw)
    # Unify various dash/hyphen characters to a plain hyphen
    text = re.sub(r"[\u2010-\u2015\u2212]", "-", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_html(html_bytes: bytes) -> str:
    """Parse HTML, strip scripts/styles, return visible text."""
    soup = BeautifulSoup(html_bytes, "html.parser")
    for tag in soup(["script", "style", "head", "meta", "link"]):
        tag.decompose()
    raw = soup.get_text(separator=" ")
    return normalize_text(raw)


def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    """Extract text from PDF bytes using pypdf (no OCR)."""
    try:
        reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))
        parts = []
        for page in reader.pages:
            page_text = page.extract_text() or ""
            parts.append(page_text)
        raw = "\n".join(parts)
        return normalize_text(raw)
    except Exception:
        return ""


print("Text extraction helpers defined.")

Text extraction helpers defined.


## 3. Document Fetching

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (research; masterarbeit) compatible",
}


def _get_with_retry(url: str) -> requests.Response | None:
    """GET a URL with simple retry logic. Returns None on failure."""
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp
            if resp.status_code in (404, 403, 410):
                return None
        except requests.RequestException:
            pass
        if attempt < MAX_RETRIES:
            time.sleep(REQUEST_DELAY * 2)
    return None


def fetch_document_text(document_url: str, document_type: str) -> tuple[str, str]:
    """
    Fetch and extract text from a document URL.

    - If document_type is 'case' or 'decision': treat as PDF.
    - Otherwise: treat as HTML.

    Returns (text, processing_status). text is empty string on failure.
    """
    time.sleep(REQUEST_DELAY)

    resp = _get_with_retry(document_url)
    if resp is None:
        return "", "fetch_failed"

    is_pdf = document_type.strip().lower() in PDF_DOCUMENT_TYPES

    if is_pdf:
        text = extract_text_from_pdf(resp.content)
        if not text or len(text) < 100:
            return "", "pdf_no_text"
        return text, "ok_pdf"
    else:
        text = extract_text_from_html(resp.content)
        if not text or len(text) < 100:
            return "", "html_no_text"
        return text, "ok_html"


print("Document fetching functions defined.")

Document fetching functions defined.


## 4. Regex Pattern Builder for EC Cases

Same logic as in `05_identify_cjeu_cases_citing_ec.ipynb`.

In [4]:
def _escape(s: str) -> str:
    """Regex-escape a string."""
    return re.escape(s)


# ── EC case-number prefix families ────────────────────────────────────────────────────────────────────────────
# All three prefixes (AT, IV, COMP) are always treated as interchangeable.
_ALL_PREFIXES = r"(?:AT|IV|COMP)"

_SEP = r"[\s./\-]*"   # flexible separator: space, dot, slash, hyphen (zero or more)


def _split_ec_case_number(ec_case_number: str) -> tuple[str, list[str]]:
    """
    Split an EC case number into (prefix, tokens).

    Examples:
      'AT.32.432'      -> ('AT',   ['32', '432'])
      'IV/34.324'      -> ('IV',   ['34', '324'])
      'COMP/38.456'    -> ('COMP', ['38', '456'])
      'COMP/D2/56.334' -> ('COMP', ['D2', '56', '334'])
      'AT/23455'       -> ('AT',   ['23455'])
    """
    m = re.match(r"^([A-Za-z]+)(.*)", ec_case_number)
    if not m:
        return "", [ec_case_number]
    prefix = m.group(1).upper()
    rest   = m.group(2)
    tokens = re.split(r"[\s./\-]+", rest.strip())
    tokens = [t for t in tokens if t]   # drop empty strings
    return prefix, tokens


def _prefix_pattern(prefix: str) -> str:
    """
    Return a regex fragment that matches all three EC case-number prefixes.
    Regardless of which prefix is stored in the master (AT, IV, or COMP),
    the pattern always matches all three alternatives.
    """
    return _ALL_PREFIXES


def _digit_flexible(token: str) -> str:
    """
    If token consists entirely of digits, return a regex fragment that allows
    an optional separator between EVERY digit. Otherwise return re.escape(token).

    Example: '31900' -> r'3[\s./\-]*1[\s./\-]*9[\s./\-]*0[\s./\-]*0'
    """
    if token.isdigit():
        return _SEP.join(re.escape(ch) for ch in token)
    return re.escape(token)


def _build_flexible_case_pattern(prefix: str, tokens: list[str]) -> str:
    """
    Build a flexible regex for an EC case number.

    - Prefix is matched via _prefix_pattern (always AT|IV|COMP).
    - Tokens are joined with a flexible separator (_SEP).
    - Pure-digit tokens additionally allow optional separators between every digit.
    - Word-boundary lookarounds prevent partial matches inside longer IDs.
    """
    if not tokens:
        return re.escape(prefix)
    token_part = _SEP.join(_digit_flexible(t) for t in tokens)
    pattern = _prefix_pattern(prefix) + _SEP + token_part
    return r"(?<![A-Za-z0-9])" + pattern + r"(?![A-Za-z0-9])"


def build_patterns_for_ec_case(row: pd.Series) -> list[dict]:
    """
    Build a list of regex patterns for a single EC antitrust case.

    Only case_number patterns (AT/IV/COMP prefix) are returned.
    No CELEX patterns, no decision-reference patterns.

    Each entry is a dict with keys:
      pattern_str    - the raw regex string
      pattern_type   - 'case_number'
      match_strength - 'medium'
    """
    patterns = []
    ec_case_number = str(row.get("ec_case_number", "")).strip()

    if ec_case_number and ec_case_number not in ("", "nan"):
        # Split on ';' to handle multiple case numbers in one cell
        sub_numbers = [s.strip() for s in ec_case_number.split(";") if s.strip()]
        for sub_number in sub_numbers:
            prefix, tokens = _split_ec_case_number(sub_number)
            if prefix and tokens:
                flex_pattern = _build_flexible_case_pattern(prefix, tokens)
                patterns.append({
                    "pattern_str":    flex_pattern,
                    "pattern_type":   "case_number",
                    "match_strength": "medium",
                })
            else:
                # Fallback: literal escape if parsing failed
                patterns.append({
                    "pattern_str":    _escape(sub_number),
                    "pattern_type":   "case_number",
                    "match_strength": "medium",
                })

    return patterns


# ── Quick smoke-tests ──────────────────────────────────────────────────────────────────────────────
print("=== _split_ec_case_number ===")
for cn in ["AT.32.432", "IV/34.324", "COMP/38.456", "COMP/D2/56.334", "AT/23455"]:
    print(f"  {cn!r:25} -> {_split_ec_case_number(cn)}")

print()
print("=== Regex match smoke-tests ===")
smoke_tests = [
    ("IV/34.324",   ["IV/34.324", "IV 34 324", "COMP/34.324", "AT.34324"]),
    ("COMP/38.456", ["IV/38.456", "COMP/38456", "AT.38456"]),
    ("AT.35814",    ["AT.35814", "IV/35.814", "COMP/35.814"]),
    ("IV/31900",    ["IV/31900", "IV/31.900", "AT/31900", "COMP/31900"]),
]
for cn, examples in smoke_tests:
    row = pd.Series({"ec_case_number": cn})
    pats = [p["pattern_str"] for p in build_patterns_for_ec_case(row)]
    for ex in examples:
        matched = any(re.search(pat, ex, re.IGNORECASE) for pat in pats)
        status = "OK" if matched else "FAIL"
        print(f"  [{status}] {cn!r} matches {ex!r}")

print()
print("Pattern builder defined.")

=== _split_ec_case_number ===
  'AT.32.432'               -> ('AT', ['32', '432'])
  'IV/34.324'               -> ('IV', ['34', '324'])
  'COMP/38.456'             -> ('COMP', ['38', '456'])
  'COMP/D2/56.334'          -> ('COMP', ['D2', '56', '334'])
  'AT/23455'                -> ('AT', ['23455'])

=== Regex match smoke-tests ===
  [OK] 'IV/34.324' matches 'IV/34.324'
  [OK] 'IV/34.324' matches 'IV 34 324'
  [OK] 'IV/34.324' matches 'COMP/34.324'
  [OK] 'IV/34.324' matches 'AT.34324'
  [OK] 'COMP/38.456' matches 'IV/38.456'
  [OK] 'COMP/38.456' matches 'COMP/38456'
  [OK] 'COMP/38.456' matches 'AT.38456'
  [OK] 'AT.35814' matches 'AT.35814'
  [OK] 'AT.35814' matches 'IV/35.814'
  [OK] 'AT.35814' matches 'COMP/35.814'
  [OK] 'IV/31900' matches 'IV/31900'
  [OK] 'IV/31900' matches 'IV/31.900'
  [OK] 'IV/31900' matches 'AT/31900'
  [OK] 'IV/31900' matches 'COMP/31900'

Pattern builder defined.


<>:44: SyntaxWarning: invalid escape sequence '\s'
<>:44: SyntaxWarning: invalid escape sequence '\s'
C:\Users\flohe\AppData\Local\Temp\ipykernel_40040\3201612515.py:44: SyntaxWarning: invalid escape sequence '\s'
  """


## 5. Self-Reference Detection

In [5]:
def get_self_case_numbers(source_row: pd.Series) -> set[str]:
    """
    Return the set of all EC case numbers that belong to the source document itself.
    Handles multiple case numbers separated by ';'.
    """
    raw = str(source_row.get("ec_case_number", "")).strip()
    if not raw or raw == "nan":
        return set()
    return {s.strip() for s in raw.split(";") if s.strip()}


print("Self-reference detection defined.")

Self-reference detection defined.


## 6. Match Extraction Function

In [6]:
def find_ec_matches_in_text(
    text: str,
    ec_master: pd.DataFrame,
    source_row: pd.Series,
    processing_status: str,
) -> list[dict]:
    """
    Search text for all EC case references defined in ec_master.

    - Excludes self-references (source document's own case numbers).
    - Deduplicates: same source + same target + same pattern_type -> keep first match only.

    Returns a list of match records.
    """
    records = []
    seen = set()

    source_case_numbers   = get_self_case_numbers(source_row)
    source_ec_case_number = str(source_row.get("ec_case_number", "")).strip()
    source_case_title     = str(source_row.get("case_title", "")).strip()
    source_date           = str(source_row.get("date", "")).strip()
    source_type           = str(source_row.get("type", "")).strip()
    source_document_type  = str(source_row.get("document_type", "")).strip()
    source_document_url   = str(source_row.get("document_url", "")).strip()

    for _, target_row in ec_master.iterrows():
        target_ec_case_number = str(target_row.get("ec_case_number", "")).strip()
        target_case_title     = str(target_row.get("case_title", "")).strip()
        target_celex_no       = str(target_row.get("celex_no", "")).strip()

        # Skip self-references
        target_sub_numbers = {s.strip() for s in target_ec_case_number.split(";") if s.strip()}
        if target_sub_numbers & source_case_numbers:
            continue

        patterns = build_patterns_for_ec_case(target_row)

        for pat_info in patterns:
            dedup_key = (source_ec_case_number, target_ec_case_number, pat_info["pattern_type"])
            if dedup_key in seen:
                continue

            try:
                compiled = re.compile(pat_info["pattern_str"], re.IGNORECASE)
            except re.error:
                continue

            match = compiled.search(text)
            if match:
                seen.add(dedup_key)
                start   = max(0, match.start() - CONTEXT_CHARS)
                end     = min(len(text), match.end() + CONTEXT_CHARS)
                context = text[start:end].replace("\n", " ")

                records.append({
                    "source_ec_case_number": source_ec_case_number,
                    "source_case_title":     source_case_title,
                    "source_date":           source_date,
                    "source_type":           source_type,
                    "source_document_type":  source_document_type,
                    "source_document_url":   source_document_url,
                    "target_ec_case_number": target_ec_case_number,
                    "target_case_title":     target_case_title,
                    "target_celex_no":       target_celex_no,
                    "matched_pattern":       pat_info["pattern_str"],
                    "matched_text":          match.group(0),
                    "match_strength":        pat_info["match_strength"],
                    "match_context":         context,
                    "processing_status":     processing_status,
                })

    return records


print("Match extraction function defined.")

Match extraction function defined.


## 7. Load Data

In [7]:
ec_master = pd.read_csv(EC_MASTER_PATH, dtype=str).fillna("")

print(f"EC antitrust master: {len(ec_master):,} rows")
print("Columns:", list(ec_master.columns))

EC antitrust master: 1,185 rows
Columns: ['ec_case_number', 'case_title', 'date', 'date_source', 'type', 'celex_no', 'document_type', 'document_url']


In [8]:
# Only process rows with a non-empty document_url
docs_to_process = ec_master[ec_master["document_url"].str.strip() != ""].copy()

# Keep only EC cases with at least a case number for the target patterns
ec_master_with_case_number = ec_master[
    ec_master["ec_case_number"].str.strip() != ""
].copy()

print(f"Documents to process (have document_url): {len(docs_to_process):,}")
print(f"EC cases with case number (targets):      {len(ec_master_with_case_number):,}")

Documents to process (have document_url): 835
EC cases with case number (targets):      1,175


## 8. Main Processing Loop

In [9]:
all_matches = []
total = len(docs_to_process)

for i, (idx, source_row) in enumerate(docs_to_process.iterrows(), start=1):
    ec_case_number = str(source_row.get("ec_case_number", "")).strip()
    document_url   = str(source_row.get("document_url", "")).strip()
    document_type  = str(source_row.get("document_type", "")).strip()

    print(f"[{i}/{total}] {ec_case_number} | {document_type} | {document_url[:80]}", end="", flush=True)

    try:
        text, processing_status = fetch_document_text(document_url, document_type)
    except Exception as e:
        print(f" -> fetch_error: {e}")
        continue

    if not text:
        print(f" -> {processing_status}")
        continue

    matches = find_ec_matches_in_text(text, ec_master_with_case_number, source_row, processing_status)
    all_matches.extend(matches)
    print(f" -> {len(matches)} match(es) [{processing_status}]")

print(f"\nDone. Total matches found: {len(all_matches):,}")

[1/835] AT.35803 | case | https://ec.europa.eu/competition/antitrust/cases1/202508/AT_35803_2560633_5_4.pd -> pdf_no_text
[2/835] AT.39294 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39294/39294_1264_5.pdf -> 0 match(es) [ok_pdf]
[3/835] AT.40166 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40166/40166_63_6.pdf -> 0 match(es) [ok_pdf]
[4/835] AT.40165 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40165/40165_66_3.pdf -> 0 match(es) [ok_pdf]
[5/835] AT.40049 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40049/40049_4172_3.pdf -> 10 match(es) [ok_pdf]
[6/835] AT.40169 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40169/40169_100_5.pdf -> 0 match(es) [ok_pdf]
[7/835] AT.40160 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40160/40160_849_7.pdf -> 0 match(es) [ok_pdf]
[8/835] AT.35814 | decision | https://ec.europa.eu/competition/antitrust/cases/de

Ignoring wrong pointing object 4 0 (offset 0)


 -> 0 match(es) [ok_pdf]
[85/835] AT.39563 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39563/39563_7279_3.pdf -> 0 match(es) [ok_pdf]
[86/835] AT.38240 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/38240/38240_28_1.pdf -> 2 match(es) [ok_pdf]
[87/835] AT.40670 | decision | https://ec.europa.eu/competition/antitrust/cases1/20263/AT_40670_18812.pdf -> 10 match(es) [ok_pdf]
[88/835] AT.39685 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39685/39685_463_6.pdf -> 0 match(es) [ok_pdf]
[89/835] AT.39686 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39686/39686_1387_5.pdf -> 0 match(es) [ok_pdf]
[90/835] AT.40797 | case | https://ec.europa.eu/competition/antitrust/cases1/202414/AT_40797_9977942_1421_3 -> pdf_no_text
[91/835] AT.40795 | decision | https://ec.europa.eu/competition/antitrust/cases1/202530/AT_40795_1262.pdf -> 0 match(es) [ok_pdf]
[92/835] AT.40432 | decision | https://ec.europa.e

invalid pdf header: b'<!DOC'
EOF marker not found


 -> pdf_no_text
[107/835] AT.40220 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40220/40220_791_5.pdf -> 0 match(es) [ok_pdf]
[108/835] AT.40588 | decision | https://ec.europa.eu/competition/antitrust/cases1/202515/AT_40588_6339.pdf -> 6 match(es) [ok_pdf]
[109/835] AT.40346 | decision | https://ec.europa.eu/competition/antitrust/cases1/202228/AT_40346_8399848_2719_3 -> 8 match(es) [ok_pdf]
[110/835] AT.40104 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40104/40104_111_7.pdf -> 0 match(es) [ok_pdf]
[111/835] AT.40105 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40105/40105_220_6.pdf -> 0 match(es) [ok_pdf]
[112/835] AT.39116 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39116/39116_2_8.pdf -> 0 match(es) [ok_pdf]
[113/835] AT.40465 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40465/40465_337_3.pdf -> 0 match(es) [ok_pdf]
[114/835] AT.40466 | decision | https:

...

 -> pdf_no_text
[258/835] AT.40966 | decision | https://ec.europa.eu/competition/antitrust/cases1/20265/AT_40966_1462.pdf -> 0 match(es) [ok_pdf]
[259/835] AT.40846 | decision | https://ec.europa.eu/competition/antitrust/cases1/202528/AT_40846_101.pdf -> 0 match(es) [ok_pdf]
[260/835] AT.37214 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/37214/37214_89_3.pdf -> 0 match(es) [ok_pdf]
[261/835] AT.38543 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/38543/38543_1489_9.pdf -> 3 match(es) [ok_pdf]
[262/835] AT.40728 | decision | https://ec.europa.eu/competition/antitrust/cases1/202445/AT_40728_10333330_2394_ -> 0 match(es) [ok_pdf]
[263/835] AT.38784 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/38784/38784_91_8.pdf -> 0 match(es) [ok_pdf]
[264/835] AT.40608 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/40608/40608_2793_5.pdf -> 0 match(es) [ok_pdf]
[265/835] AT.38662 | decision | https://ec.

invalid pdf header: b'<!DOC'
EOF marker not found


 -> pdf_no_text
[295/835] AT.39775 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39775/39775_505_8.pdf -> 0 match(es) [ok_pdf]
[296/835] AT.39654 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39654/39654_2861_16.pd -> 0 match(es) [ok_pdf]
[297/835] AT.39414 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39414/39414_171_3.pdf -> 0 match(es) [ok_pdf]
[298/835] AT.40763 | case | https://ec.europa.eu/competition/antitrust/cases1/202324/AT_40763_9237883_689_7. -> pdf_no_text
[299/835] AT.40642 | decision | https://ec.europa.eu/competition/antitrust/cases1/202318/AT_40642_9175396_2354_3 -> 0 match(es) [ok_pdf]
[300/835] AT.39899 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39899/39899_864_3.pdf -> 0 match(es) [ok_pdf]
[301/835] AT.39779 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39779/39779_417_5.pdf -> pdf_no_text
[302/835] AT.39416 | decision | https://ec.europa.eu/compet

Ignoring wrong pointing object 4 0 (offset 0)


 -> 0 match(es) [ok_pdf]
[443/835] AT.38863 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/38863/38863_46_2.pdf -> 0 match(es) [ok_pdf]
[444/835] AT.38620 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/38620/38620_880_14.pdf -> 8 match(es) [ok_pdf]
[445/835] AT.37773 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/37773/37773_142_1.pdf -> 1 match(es) [ok_pdf]
[446/835] AT.35473 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/35473/35473_11_3.pdf -> 0 match(es) [ok_pdf]
[447/835] AT.35470 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/35470/35470_14_4.pdf -> 0 match(es) [ok_pdf]
[448/835] AT.39942 | case | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39942/39942_66_5.pdf -> pdf_no_text
[449/835] AT.39822 | decision | https://ec.europa.eu/competition/antitrust/cases/dec_docs/39822/39822_8681_3.pdf -> 0 match(es) [ok_pdf]
[450/835] AT.39824 | decision | https://ec

## 9. Build Results DataFrame and Export

In [10]:
OUTPUT_COLUMNS = [
    "source_ec_case_number",
    "source_case_title",
    "source_date",
    "source_type",
    "source_document_type",
    "source_document_url",
    "target_ec_case_number",
    "target_case_title",
    "target_celex_no",
    "matched_pattern",
    "matched_text",
    "match_strength",
    "match_context",
    "processing_status",
]

if all_matches:
    results_df = pd.DataFrame(all_matches, columns=OUTPUT_COLUMNS)
else:
    results_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

print(f"Result rows: {len(results_df):,}")
results_df.head()

Result rows: 480


,source_ec_case_number,source_case_title,source_date,source_type,source_document_type,source_document_url,target_ec_case_number,target_case_title,target_celex_no,matched_pattern,matched_text,match_strength,match_context,processing_status
0,AT.40049,MasterCard inter-regional MIF monitoring,2019-04-29,AtStandardATCCase,decision,https://ec.europa.eu/competition/antitrust/cas...,AT.40153,E-book MFNs and related matters (Amazon),,(?<![A-Za-z0-9])(?:AT|IV|COMP)[\s./\-]*4[\s./\...,AT.40153,medium,EEA as “those countries participating in the E...,ok_pdf
1,AT.40049,MasterCard inter-regional MIF monitoring,2019-04-29,AtStandardATCCase,decision,https://ec.europa.eu/competition/antitrust/cas...,AT.29373,Visa International,,(?<![A-Za-z0-9])(?:AT|IV|COMP)[\s./\-]*2[\s./\...,IV/29.373,medium,to be paid in full in each statement cycle. 7...,ok_pdf
2,AT.40049,MasterCard inter-regional MIF monitoring,2019-04-29,AtStandardATCCase,decision,https://ec.europa.eu/competition/antitrust/cas...,AT.39398,Visa Inter-Regional MIF monitoring,,(?<![A-Za-z0-9])(?:AT|IV|COMP)[\s./\-]*3[\s./\...,AT.39398,medium,Mastercard consumer c redit card that is non -...,ok_pdf
3,AT.40049,MasterCard inter-regional MIF monitoring,2019-04-29,AtStandardATCCase,decision,https://ec.europa.eu/competition/antitrust/cas...,AT.39847,Ebooks,,(?<![A-Za-z0-9])(?:AT|IV|COMP)[\s./\-]*3[\s./\...,AT.39847,medium,T.39767 - BEH Electricity; Commission Decision...,ok_pdf
4,AT.40049,MasterCard inter-regional MIF monitoring,2019-04-29,AtStandardATCCase,decision,https://ec.europa.eu/competition/antitrust/cas...,AT.39767,BEH Electricity,,(?<![A-Za-z0-9])(?:AT|IV|COMP)[\s./\-]*3[\s./\...,AT.39767,medium,th a duration of five years see for example: C...,ok_pdf


In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(results_df):,} rows to: {OUTPUT_PATH}")

Saved 480 rows to: data\processed\ec_ec_case_matches.csv


## 10. Summary Statistics

In [12]:
if not results_df.empty:
    print("=== Match Strength Distribution ===")
    print(results_df["match_strength"].value_counts().to_string())
    print()
    print("=== Processing Status Distribution ===")
    print(results_df["processing_status"].value_counts().to_string())
    print()
    print("=== Top 10 Most-Cited Target EC Cases ===")
    print(
        results_df.groupby(["target_ec_case_number", "target_case_title"])
        .size()
        .sort_values(ascending=False)
        .head(10)
        .to_string()
    )
    print()
    print("=== Top 10 Source EC Documents with Most Citations ===")
    print(
        results_df.groupby("source_ec_case_number")["target_ec_case_number"]
        .nunique()
        .sort_values(ascending=False)
        .head(10)
        .to_string()
    )
else:
    print("No matches found.")

=== Match Strength Distribution ===
match_strength
medium    480

=== Processing Status Distribution ===
processing_status
ok_pdf    480

=== Top 10 Most-Cited Target EC Cases ===
target_ec_case_number  target_case_title               
IV/31865               PVC                                 12
AT.35691               Pre-insulated pipe cartel           10
AT.38281               Raw Tobacco IT                       9
IV/3344; IV/4          Grundig-Consten                      9
IV/4                   GRUNDIG                              9
AT.38233               Wanadoo                              8
AT.37451               Price squeeze local loop Germany     8
AT.39462               Freight Forwarding                   6
IV/31149               POLYPROPYLENE                        6
AT.39740               Google Search (Shopping)             6

=== Top 10 Source EC Documents with Most Citations ===
source_ec_case_number
AT.38645    19
AT.39437    19
AT.38589    18
AT.40324    13
AT.398